# AI기초 12주차 실습 — 트랜스포머를 직접 뜯어보기

이론 시간에 배운 세 가지를 **코드로 확인**합니다. 모델을 잘 만드는 것보다, 숫자를 직접 보며 *왜 그렇게 되는지* 이해하는 것이 목표입니다.

| 실습 | 확인할 것 | 이론 슬라이드 |
|---|---|---|
| 1. 셀프 어텐션과 멀티 헤드 | 손계산 = 코드 결과, √dₖ의 이유, 멀티 헤드의 모양 | 9~15번 |
| 2. 포지셔널 인코딩 | 어텐션은 순서를 모른다 → 위치 벡터를 더하면 구별한다 | 19~22번 |
| 3. 트랜스포머 블록 | 블록 조립, 마스크, 위치 정보가 정말 필요한지 실험 | 24~29번 |

**실행 방법:** Colab에서 `런타임 → 모두 실행`. GPU가 없어도 전체 2~3분이면 끝납니다.

## 0. 준비

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # 텐서플로 경고 메시지 숨기기

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import keras
from keras import layers

# 그래프에 한글이 보이도록 글꼴 설정 (Colab이면 나눔고딕을 설치)
if os.path.exists('/content'):
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for path in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
             '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    if os.path.exists(path):
        fm.fontManager.addfont(path)
        plt.rcParams['font.family'] = fm.FontProperties(fname=path).get_name()
        break
plt.rcParams['axes.unicode_minus'] = False

np.set_printoptions(precision=3, suppress=True)
keras.utils.set_random_seed(42)   # 실행할 때마다 같은 결과가 나오도록
print('Keras', keras.__version__)

---
# 실습 1. 셀프 어텐션과 멀티 헤드 어텐션

## 1-1. 이론 10번 슬라이드의 손계산을 코드로

“고양이가 생선을 먹었다”에서 ‘먹었다’가 셀프 어텐션을 하는 장면입니다. 숫자는 슬라이드와 똑같습니다.
아래 네 줄이 **① 점수 → ② ÷√dₖ → ③ softmax → ④ 가중합**에 하나씩 대응합니다.

In [ ]:
words = ['고양이가', '생선을', '먹었다']

q = np.array([2, 0, 2, 0])              # '먹었다'의 Query (= x·W_Q)
K = np.array([[1, 0, 1, 0],             # 각 단어의 Key  (= x·W_K)
              [1, 1, 0, 0],
              [0, 1, 0, 1]])
V = np.array([[10, 0],                  # 각 단어의 Value (= x·W_V)
              [0, 10],
              [5, 5]])
d_k = K.shape[1]                        # Key의 차원 = 4

scores  = K @ q                                   # ① 점수: q와 모든 Key의 내적
scaled  = scores / np.sqrt(d_k)                   # ② √d_k(=2)로 나누기
weights = np.exp(scaled) / np.exp(scaled).sum()   # ③ softmax → 합이 1인 가중치
output  = weights @ V                             # ④ Value의 가중합

for w, s, sc, wt in zip(words, scores, scaled, weights):
    print(f'{w:5s}  ① 점수 {s:2d}   ② ÷2 → {sc:.1f}   ③ 가중치 {wt:.3f}')
print('가중치 합 =', weights.sum().round(3))
print('④ 출력 =', output.round(2))

✅ **확인:** 가중치 0.665 / 0.245 / 0.090, 출력 [7.1, 2.9] — 슬라이드의 손계산과 같습니다.
자기 자신(‘먹었다’)에게도 0.090의 가중치가 간다는 점도 보세요.

## 1-2. 행렬로 한 번에: 문장 전체의 셀프 어텐션

실제로는 단어 하나씩이 아니라 **문장 전체를 행렬로 한 번에** 계산합니다.

`Attention(Q, K, V) = softmax(QKᵀ / √dₖ) V`,  `Q = XW_Q`, `K = XW_K`, `V = XW_V`

이번에는 단어 임베딩 X와 가중치 행렬 W를 무작위로 만듭니다(학습 전 상태라고 생각하세요).

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)    # 큰 수 때문에 계산이 터지지 않게 하는 요령
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)  # 행마다 합이 1

def self_attention(X, W_Q, W_K, W_V):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V       # 한 단어에서 세 벡터 만들기 (8번 슬라이드)
    d_k = K.shape[-1]
    weights = softmax(Q @ K.T / np.sqrt(d_k))  # n × n 가중치 표
    return weights @ V, weights, Q, K

rng = np.random.default_rng(0)
n, d_model, d_k = 3, 6, 4                    # 단어 3개, 임베딩 6차원, Q·K·V 4차원
X   = rng.normal(size=(n, d_model))          # 단어 임베딩 (행 하나 = 단어 하나)
W_Q = rng.normal(size=(d_model, d_k)) * 0.5   # 학습 전의 작은 무작위 값
W_K = rng.normal(size=(d_model, d_k)) * 0.5
W_V = rng.normal(size=(d_model, d_k)) * 0.5

out, A, Q, K = self_attention(X, W_Q, W_K, W_V)
print('X 모양     :', X.shape)
print('가중치 표  :', A.shape, '\n', A)
print('행마다 합  :', A.sum(axis=1))
print('출력 모양  :', out.shape)
print()
print("'먹었다'의 q =", Q[2])
print("'먹었다'의 k =", K[2], ' ← 같은 단어라도 q와 k는 다르다')

✅ **확인:** 가중치 표는 3 × 3(단어 수 × 단어 수)이고 **행마다 합이 1**입니다.
같은 단어에서 나온 q와 k가 다른 것은, 같은 x에 **서로 다른 행렬**(W_Q, W_K)을 곱했기 때문입니다.

## 1-3. 왜 √dₖ로 나눌까? (이론 12번 슬라이드)

차원 d가 커지면 내적이 커지고, softmax가 한 단어에 몰립니다. 무작위 Query 1,000개가 Key 10개를 볼 때,
**가장 큰 가중치의 평균**이 얼마인지 √d로 나누기 전과 후를 비교합니다. (1에 가까울수록 한 단어에 몰린 것)

In [ ]:
rng = np.random.default_rng(1)
print(' 차원 d | 나누기 전 최대 가중치 | √d로 나눈 후')
for d in [4, 64, 512]:
    q = rng.normal(size=(1000, d))       # Query 1000개
    k = rng.normal(size=(10, d))         # Key 10개
    raw = q @ k.T                        # 점수 (1000 × 10)
    before = softmax(raw).max(axis=1).mean()
    after  = softmax(raw / np.sqrt(d)).max(axis=1).mean()
    print(f'{d:6d}  |        {before:.2f}          |     {after:.2f}')

✅ **확인:** 나누기 전에는 d가 클수록 최대 가중치가 1에 가까워집니다(한 단어가 전부 가져감 → 기울기 ≈ 0).
√d로 나누면 d와 상관없이 비슷한 수준으로 유지됩니다.

## 1-4. 멀티 헤드 어텐션: Keras로 모양 확인하기 (이론 15번 슬라이드)

Keras의 `MultiHeadAttention`을 씁니다. 헤드 2개, 헤드마다 `key_dim=4`.
셀프 어텐션이므로 **같은 X를 query와 value 자리에 모두** 넣습니다.

In [ ]:
mha = layers.MultiHeadAttention(num_heads=2, key_dim=4)
X_in = X[None].astype('float32')            # Keras는 (배치, 단어 수, 차원) 모양을 원함 → (1, 3, 6)
out, scores = mha(X_in, X_in, return_attention_scores=True)

print('입력 모양          :', X_in.shape)
print('출력 모양          :', tuple(out.shape), ' ← 입력과 같은 크기')
print('가중치 표 모양     :', tuple(scores.shape), ' ← (배치, 헤드 수, n, n)')
print()
print('헤드별 행렬 모양:')
for w in mha.weights:
    if 'kernel' in w.path:
        print(f'  {w.path.split("/")[-2]:7s}', tuple(w.shape))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for h in range(2):
    ax = axes[h]
    ax.imshow(scores[0, h], cmap='Oranges', vmin=0, vmax=1)
    ax.set_title(f'헤드 {h+1}')
    ax.set_xticks(range(3), words); ax.set_yticks(range(3), words)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{float(scores[0, h, i, j]):.2f}', ha='center', va='center')
plt.suptitle('같은 문장, 다른 헤드 → 다른 가중치 표 (학습 전)')
plt.tight_layout(); plt.show()

✅ **확인:** 헤드마다 자기만의 W_Q·W_K·W_V(`(6, 2, 4)` = 입력 6차원 → 헤드 2개 × 4차원)를 가지므로
같은 문장을 보고도 **서로 다른 가중치 표**를 만듭니다. 지금은 학습 전이라 값에 의미는 없지만,
학습하면 헤드마다 다른 관계(주어, 목적어 등)를 맡게 됩니다(16번 슬라이드).

---
# 실습 2. 포지셔널 인코딩

## 2-1. 셀프 어텐션은 정말 순서를 모를까? (이론 19번 슬라이드)

“dog bites man”과 “man bites dog”를 같은 어텐션 층에 넣고, 두 문장에서 **‘dog’의 출력 벡터**를 비교합니다.

In [ ]:
rng = np.random.default_rng(2)
emb = {w: rng.normal(size=8).astype('float32') for w in ['dog', 'bites', 'man']}   # 단어 임베딩 (8차원)

sent1 = ['dog', 'bites', 'man']
sent2 = ['man', 'bites', 'dog']
X1 = np.stack([emb[w] for w in sent1])[None]   # (1, 3, 8)
X2 = np.stack([emb[w] for w in sent2])[None]

att = layers.MultiHeadAttention(num_heads=2, key_dim=4)
out1 = att(X1, X1).numpy()[0]
out2 = att(X2, X2).numpy()[0]

dog1 = out1[sent1.index('dog')]
dog2 = out2[sent2.index('dog')]
print("문장1의 'dog' 출력:", dog1[:4], '...')
print("문장2의 'dog' 출력:", dog2[:4], '...')
print('완전히 같은가?', np.allclose(dog1, dog2, atol=1e-5))

✅ **확인:** `True` — 순서가 바뀌어도 ‘dog’의 결과가 똑같습니다. 어텐션에게 두 문장은 같은 **단어 묶음**일 뿐입니다.

## 2-2. 사인·코사인 포지셔널 인코딩 만들기 (이론 21~22번 슬라이드)

`PE(pos, 2i) = sin(pos / 10000^(2i/d))`, `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`

In [ ]:
def positional_encoding(n, d):
    pos = np.arange(n)[:, None]                  # 위치 0, 1, ..., n-1  (세로)
    i   = np.arange(d)[None, :]                  # 차원 0, 1, ..., d-1  (가로)
    angle = pos / np.power(10000, (2 * (i // 2)) / d)
    return np.where(i % 2 == 0, np.sin(angle), np.cos(angle)).astype('float32')

pe = positional_encoding(50, 64)
print('PE 모양:', pe.shape, ' (위치 50개 × 차원 64)')
print('              차원0   차원16  차원40')
for p in [10, 12, 30]:
    print(f'위치 {p:2d} :', pe[p, [0, 16, 40]])

✅ **확인:** 위치 10의 [−0.54, 0.84, 0.03]은 21번 슬라이드 표와 같습니다.
위치 10과 12는 **차원 0(초침)의 값이 같지만**, 차원 16(분침)에서 0.84와 0.93으로 갈립니다 — 여러 바늘을 함께 읽어야 자리가 정해집니다.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.imshow(pe, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
plt.xlabel('차원'); plt.ylabel('위치'); plt.colorbar(label='값')
plt.title('포지셔널 인코딩: 가로줄 하나 = 한 위치의 위치 벡터')
plt.show()

이제 2-1의 두 문장에 위치 벡터를 **더한 뒤** 다시 비교합니다.

In [ ]:
pe8 = positional_encoding(3, 8)          # 단어 3개, 8차원 (임베딩과 같은 크기)
out1 = att(X1 + pe8, X1 + pe8).numpy()[0]
out2 = att(X2 + pe8, X2 + pe8).numpy()[0]

dog1 = out1[sent1.index('dog')]
dog2 = out2[sent2.index('dog')]
print("문장1의 'dog' 출력:", dog1[:4], '...')
print("문장2의 'dog' 출력:", dog2[:4], '...')
print('완전히 같은가?', np.allclose(dog1, dog2, atol=1e-5))

✅ **확인:** `False` — 0번 자리의 dog와 2번 자리의 dog가 이제 구별됩니다. 어텐션 층은 그대로이고, **입력에 위치 벡터를 더한 것**만 바꿨습니다.

---
# 실습 3. 트랜스포머 블록 조립하기

## 3-1. 인코더 블록 (이론 25~26번 슬라이드)

`멀티 헤드 셀프 어텐션 → Add & Norm → 피드포워드 → Add & Norm`을 그대로 코드로 옮깁니다.

In [ ]:
def encoder_block(x, num_heads=2, key_dim=8, ff_dim=32):
    # ① 멀티 헤드 셀프 어텐션 (회의)
    att_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    # ③ Add & Norm: 입력 x를 지름길로 더하고(잔차 연결) 정규화
    x = layers.LayerNormalization()(x + att_out)
    # ④ 피드포워드 (각자 정리): 넓혔다가 원래 크기로
    ff_out = layers.Dense(ff_dim, activation='relu')(x)
    ff_out = layers.Dense(x.shape[-1])(ff_out)
    # ③ Add & Norm
    return layers.LayerNormalization()(x + ff_out)

inp = keras.Input(shape=(8, 16))          # 단어 8개, 16차원
h1 = encoder_block(inp)
h2 = encoder_block(h1)                    # 블록을 한 층 더 쌓기
print('입력        :', inp.shape)
print('블록 1 출력 :', h1.shape)
print('블록 2 출력 :', h2.shape, ' ← 입력과 크기가 같아서 계속 쌓을 수 있다')
print('2층 인코더의 학습 파라미터 수:', keras.Model(inp, h2).count_params())

✅ **확인:** 블록을 통과해도 크기가 (단어 수, 16)으로 그대로입니다. 그래서 레고처럼 N층을 쌓을 수 있습니다.

## 3-2. 디코더의 마스크: 미래를 가린다 (이론 27번 슬라이드)

`use_causal_mask=True` 한 줄이면 미래 칸의 가중치가 0이 됩니다.

In [ ]:
tokens = ['<s>', 'I', 'am', 'a', 'student']
rng = np.random.default_rng(3)
Y = rng.normal(size=(1, 5, 8)).astype('float32')

masked_att = layers.MultiHeadAttention(num_heads=1, key_dim=8)
_, w = masked_att(Y, Y, use_causal_mask=True, return_attention_scores=True)
w = w.numpy()[0, 0]                       # (5, 5) 가중치 표
print(w)
print('행마다 합:', w.sum(axis=1))

plt.figure(figsize=(4.5, 4))
plt.imshow(w, cmap='Oranges', vmin=0, vmax=1)
plt.xticks(range(5), tokens); plt.yticks(range(5), tokens)
plt.xlabel('보이는 단어 (Key)'); plt.ylabel('보는 단어 (Query)')
plt.title('마스크드 셀프 어텐션'); plt.show()

✅ **확인:** 대각선 위쪽(미래)이 모두 **0**이고, 각 행은 여전히 합이 1입니다.
‘am’ 행은 `<s>`, `I`, `am`에만 가중치를 나눠 줍니다.

## 3-3. 종합 실험: 위치 정보가 정말 필요할까?

간단한 과제를 학습시켜 봅니다. 숫자 8개로 된 수열에 **A(=1)와 B(=2)가 하나씩** 들어 있고, 나머지는 3~9 사이의 숫자입니다.

> **과제:** A가 B보다 **앞에** 있으면 1, 아니면 0

같은 인코더 블록 모델을 **포지셔널 인코딩 없이 / 있게** 두 번 학습시켜 정확도를 비교합니다.

In [ ]:
SEQ_LEN, VOCAB, A, B = 8, 10, 1, 2

def make_data(n, rule, seed):
    rng = np.random.default_rng(seed)
    X = rng.integers(3, VOCAB, size=(n, SEQ_LEN))         # 3~9로 채우고
    for row in X:
        a, b = rng.choice(SEQ_LEN, 2, replace=False)       # 서로 다른 두 자리에
        row[a], row[b] = A, B                              # A와 B를 하나씩 넣기
    y = np.array([rule(list(x)) for x in X], dtype='float32')
    return X, y

# ★ 정답 규칙: A가 B보다 앞에 있는가?
rule = lambda x: x.index(A) < x.index(B)

X_demo, y_demo = make_data(3, rule, seed=9)
for x, t in zip(X_demo, y_demo):
    print(x, '→ 정답', int(t))

In [ ]:
class AddPositionalEncoding(layers.Layer):
    def call(self, x):
        return x + positional_encoding(x.shape[1], x.shape[2])

def build_model(use_pe):
    inp = keras.Input(shape=(SEQ_LEN,), dtype='int32')
    x = layers.Embedding(VOCAB, 16)(inp)           # 단어 임베딩
    if use_pe:
        x = AddPositionalEncoding()(x)              # ← 차이는 이 한 줄뿐
    x = encoder_block(x)                            # 3-1에서 만든 인코더 블록
    x = layers.GlobalAveragePooling1D()(x)          # 단어 벡터들의 평균
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def run_experiment(rule):
    X_train, y_train = make_data(8000, rule, seed=0)
    X_test,  y_test  = make_data(2000, rule, seed=1)
    result = {}
    for use_pe in [False, True]:
        keras.utils.set_random_seed(42)
        model = build_model(use_pe)
        model.fit(X_train, y_train, epochs=15, batch_size=64, verbose=0)
        acc = model.evaluate(X_test, y_test, verbose=0)[1]
        name = 'PE 있음' if use_pe else 'PE 없음'
        result[name] = acc
        print(f'{name}: 테스트 정확도 {acc:.1%}')
    return result

result = run_experiment(rule)

✅ **확인:** PE가 없으면 정확도가 **약 50%** — 동전 던지기 수준입니다. 모델 입장에서는 [A, …, B]와 [B, …, A]가 같은 단어 묶음이라 구별할 방법이 없습니다.
PE를 더하는 **한 줄**만 추가하면 거의 100%가 됩니다.

---
# 📝 과제

3-3의 실험에서 **`rule` 한 줄만 바꿔** 아래 두 규칙으로 실험하세요. 나머지 코드는 그대로 둡니다.

| 규칙 | 설명 |
|---|---|
| 규칙 ① | 수열에 숫자 **3**이 하나라도 들어 있으면 1 |
| 규칙 ② | **A가 앞쪽 절반**(0~3번 자리)에 있으면 1 |

**제출할 것**
1. 아래 표를 채우세요 (3-3의 원래 규칙 포함, 정확도 6개).
2. 결과를 보고 **3~4문장**으로 설명하세요: PE가 없어도 잘 되는 규칙은 무엇이고, 왜 그런가? PE가 꼭 필요한 규칙은 무엇이고, 왜 그런가?

| 규칙 | PE 없음 | PE 있음 |
|---|---|---|
| A가 B보다 앞 (3-3) | | |
| ① 3이 들어 있음 | | |
| ② A가 앞쪽 절반 | | |

⚠️ 빈칸(`...`)을 채우기 전에는 아래 두 셀에서 오류가 나는 것이 정상입니다.

💡 힌트: `x`는 숫자 8개짜리 리스트입니다. `3 in x`, `x.index(A)`, `SEQ_LEN // 2`를 활용해 보세요.

In [ ]:
# 규칙 ①: 수열에 3이 들어 있는가?
rule1 = lambda x: ...   # TODO: 이 줄을 완성하세요
result1 = run_experiment(rule1)

In [ ]:
# 규칙 ②: A가 앞쪽 절반(0~3번 자리)에 있는가?
rule2 = lambda x: ...   # TODO: 이 줄을 완성하세요
result2 = run_experiment(rule2)

**설명 (3~4문장):** 여기에 작성하세요.

---
### 오늘 확인한 것
1. 셀프 어텐션 = 점수 → ÷√dₖ → softmax → 가중합. 손계산과 코드 결과가 같다.
2. 어텐션만으로는 순서를 모른다. 위치 벡터를 **더하면** 구별한다.
3. 블록은 입력과 출력 크기가 같아 쌓을 수 있고, 마스크는 미래 칸의 가중치를 0으로 만든다.